<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [1]:
# TODO 0: 실습을 위해 아래 패키지를 import하고, diamonds 데이터셋을 준비해주세요.
import numpy as np
import seaborn as sns
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

# diamonds 데이터셋 로드 (carat → price 예측)
df = sns.load_dataset("diamonds")
X = df[["carat"]].values
y = df["price"].values

# 표준화
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
y_mean, y_std = y.mean(), y.std()
X_scaled = (X - X_mean) / X_std
y_scaled = (y - y_mean) / y_std

# 바이어스 항 추가
X_b = np.c_[np.ones((len(X_scaled), 1)), X_scaled]

# Train/Valid 분할 (80:20)
dataset_count = len(X_b)
shuffled_index = np.random.permutation(dataset_count)
cut_index = int(dataset_count * 0.8)
X_train, X_valid = X_b[shuffled_index[:cut_index]], X_b[shuffled_index[cut_index:]]
y_train, y_valid = y_scaled[shuffled_index[:cut_index]], y_scaled[shuffled_index[cut_index:]]

# 비용 함수 및 초기화
def mean_squared_error(X, y, theta):
    data_count = len(y)
    predictions = X @ theta
    cost = (1 / data_count) * np.sum((y - predictions) ** 2)

    return cost

theta = np.zeros(X_b.shape[1])
learning_rate = 0.01

print(f"데이터: diamonds ({len(X)}개, carat → price)")
print(f"Train: {len(X_train)}개, Valid: {len(X_valid)}개")

데이터: diamonds (53940개, carat → price)
Train: 43152개, Valid: 10788개


</br>

# 학습 내용
>이번 장에서는 <strong>미니배치 학습(Mini-batch Learning)</strong>에 대해 학습합니다.</br></br>
>배치 크기에 따른 학습 전략 차이와 조기 종료 기법을 학습해봅시다.

</br>

# 경사하강법의 변형 (GD Variants)
> 한 번의 파라미터 업데이트에 사용하는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">데이터 양에 따라</mark> 세 가지 변형이 있습니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">방법</th>
      <th style="text-align:center">배치 크기</th>
      <th>장점</th>
      <th>단점</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Batch GD</td><td style="text-align:center">전체 데이터</td><td>안정적 수렴</td><td>느림, 메모리 부족</td></tr>
    <tr><td style="text-align:center">SGD</td><td style="text-align:center">1개</td><td>빠름</td><td>불안정, 진동</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Mini-batch GD</mark></td><td style="text-align:center">n개 (32~256)</td><td>속도와 안정성 균형</td><td>배치 크기 조정 필요</td></tr>
  </tbody>
</table>

</br>

In [4]:
# TODO 1: epochs는 100, batch_size는 32를 저장해봅시다.

epochs = 100
batch_size = 32

In [6]:
# TODO 2: train_data_count에 학습용 데이터 개수, batches에 batch의 개수를 저장해봅시다.
train_data_count = len(X_train)
batches = train_data_count // batch_size

print(train_data_count)
print(batches)

43152
1348


In [ ]:
# TODO 3: epochs 만큼 학습을 진행해봅시다.

for epoch in range(epochs):

    # TODO 3-1: epoch 마다 cost를 저장하기 위한 변수를 선언해봅시다.
    epoch_cost = 0
    
    # TODO 3-2: X_train와 y_train을 무작위로 섞어 X_shuffled, y_shuffled에 저장해봅시다.
    train_shuffled_index = np.random.permutation(train_data_count)
    X_shuffled = X_train[train_shuffled_index]
    y_shuffled = y_train[train_shuffled_index]

    # TODO 3-3: 배치 개수만큼 반복하여 cost를 계산해봅시다.
    for batch in range(batches):

        # TODO 3-3-1: 각 배치의 시작과 끝 인덱스를 batch_start_index, batch_end_index에 저장해봅시다.
        batch_start_index = batch * batch_size
        batch_end_index = batch_start_index + batch_size
        
        # TODO 3-3-2: X_shuffled와 y_shuffled를 배치 크기 단위로 나눠 X_batch, y_batch에 저장해봅시다.
        X_batch = X_shuffled[batch_start_index:batch_end_index]
        y_batch = y_shuffled[batch_start_index:batch_end_index]

        # TODO 3-3-3: gradient에 기울기를 저장해봅시다.
        #             주의: batch_size 단위로 계산되어야합니다.
        
        gradient = (2/batch_size) * X_batch.T @ (X_batch @ theta -y_batch)

        # TODO 3-3-4: theta를 업데이트해봅시다.
        #             learning_rate는 TODO 0에 정의되어있습니다.
        theta = theta - learning_rate* gradient

        # TODO 3-3-5: epoch_cost에 현재 batch의 cost를 더해봅시다.
        epoch_cost += mean_squared_error(X_batch, y_batch, theta)

    # TODO 3-4: 20 epoch 마다 평균 비용을 출력해봅시다.
    if epoch % 20 == 0:
        print(epoch, epoch_cost/batches)

0 0.1647819330286458
20 0.14937007005877967
40 0.1494535914232225
60 0.149368339135305
80 0.1493798684701228


</br>

## 배치 크기의 영향 (Batch Size)

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">배치 크기</th>
      <th style="text-align:center">그래디언트 노이즈</th>
      <th style="text-align:center">수렴 속도</th>
      <th style="text-align:center">일반화</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">작음 (16~32)</td><td style="text-align:center">높음</td><td style="text-align:center">느림</td><td style="text-align:center">좋음</td></tr>
    <tr><td style="text-align:center">큼 (256~1024)</td><td style="text-align:center">낮음</td><td style="text-align:center">빠름</td><td style="text-align:center">나쁠 수 있음</td></tr>
  </tbody>
</table>

💡배치 크기 선택
> 일반적으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">32 또는 64</mark>로 시작합니다.</br>
> GPU 메모리가 허용하면 크기를 늘려 학습 속도를 높일 수 있습니다.

</br>

# 조기 종료 (Early Stopping)
> 검증 손실이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">더 이상 개선되지 않으면 학습을 중단</mark>하여 과적합을 방지합니다.

In [ ]:
# TODO 4:
#  - best_validation_loss에 최적 검증 손실 초기값을 무한대로 저장해봅시다.
#  - patience에 성능 개선 없이 허용하는 에폭 횟수 5를 저장해봅시다.
#  - no_improve에 개선 없는 횟수를 추적할 변수를 선언해봅시다.
#  - min_delta에 최소 개선 변화량 0.0001을 저장해봅시다.

In [ ]:
# TODO 5:
#  - max_epochs에 최대 에포크 수 200을 저장해봅시다.
#  - batch_size에 32을 저장해봅시다.
#  - batches에 batch의 개수를 저장해봅시다.
#  - theta를 초기화해봅시다.

In [ ]:
# TODO 6: 최대 에폭만큼 학습을 진행해봅시다.

for epoch in range(max_epochs):

    # TODO 6-1: 미니배치 학습을 진행해봅시다.
    #           (TODO 3의 학습 루프와 동일합니다.)

    for batch in range(len(X_train) // batch_size):

    # TODO 6-2: validation_loss 검증 손실, train_loss에 학습 손실을 저장해봅시다.

    # TODO 6-3: 검증 손실이 개선되었으면 best_validation_loss 갱신하고 no_improve를 초기화,
    #           개선되지 않았으면 no_improve를 1 증가시켜봅시다.
    if validation_loss < best_validation_loss - min_delta:
    else:

    # TODO 6-4: 10 epoch 마다 학습 손실과 검증 손실을 출력해봅시다.
    if epoch % 10 == 0:

    # TODO 6-5: no_improve가 patience 이상이면 학습을 중단해봅시다.
    if no_improve >= patience: